In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema import Document
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)

True

In [2]:
# Load the text file
loader = TextLoader("../data/text_files/python_intro.txt")
raw_docs = loader.load()

# Split text into documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.split_documents(raw_docs)

docs

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='backward-compatible with earlier versions. Beginning with Python 3.5,[39] ca

In [3]:
# User query
query = "How can i use python to build AI applications?"

In [4]:
### FAISS and Huggingface model Embedding
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k":8})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11544.68it/s]


In [5]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x13affd550>, search_kwargs={'k': 8})

In [6]:
llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [ ]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevent to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevence of each document to the users question.
- Return a list fo document indices in ranked order, starting from the most relevent.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [25]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='f0e6e5b8-2966-46d5-bcdb-433c2d6b2ab1', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and'),
 Document(id='d1e1d25c-c41a-40f9-9dee-df432f178fe8', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive'),
 Document(id='56f04595-4d54-4821-ad41-c45c00b86fd0', metadata={'source': '../data/text_files/python_intro.txt'}, page_content="the project's chief decision-maker.[45][c] In January 2019, active Python core developers elected a five-member Steering Council to lead the project.[46][47]"),
 Document(id='d575bafd-7166-442d-b94a-b9d213881a5a', metadata={'source': '../dat

In [26]:
chain = prompt | llm | StrOutputParser()

chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevent to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevence of each document to the users question.\n- Return a list fo documents indices in ranked order, starting from the most relevent.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser()

In [27]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [28]:
doc_lines

['1. Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and',
 '2. Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive',
 "3. the project's chief decision-maker.[45][c] In January 2019, active Python core developers elected a five-member Steering Council to lead the project.[46][47]",
 '4. Python was conceived in the late 1980s[11] by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands.[43] It was designed as a successor to the ABC programming language, which',
 "5. and logic programming.[61] Python is often referred to as a 'glue language'[62] because it is purposely designed to be able to integrate components written in other languages.",
 '6. Nevertheless, rather than building all functionality into

In [29]:
print(formatted_docs)

1. Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and
2. Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive
3. the project's chief decision-maker.[45][c] In January 2019, active Python core developers elected a five-member Steering Council to lead the project.[46][47]
4. Python was conceived in the late 1980s[11] by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands.[43] It was designed as a successor to the ABC programming language, which
5. and logic programming.[61] Python is often referred to as a 'glue language'[62] because it is purposely designed to be able to integrate components written in other languages.
6. Nevertheless, rather than building all functionality into its core, Python was 

In [ ]:
response = chain.invoke({"question": query, "documents": formatted_docs})
response

'6,5,8,1,2,7,4,3'

In [31]:
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]

indices

[5, 4, 7, 0, 1, 6, 3, 2]

In [32]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='9c75aa4a-e892-4893-9df1-4cc57af65f7d', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Nevertheless, rather than building all functionality into its core, Python was designed to be highly extensible through modules. This compact modularity has made it particularly popular as a means of'),
 Document(id='69d9592e-b4bf-4abb-98b1-07a794a31c01', metadata={'source': '../data/text_files/python_intro.txt'}, page_content="and logic programming.[61] Python is often referred to as a 'glue language'[62] because it is purposely designed to be able to integrate components written in other languages."),
 Document(id='28e83e09-a02f-4595-82fd-30c6702758b7', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'),
 Document(id='f0e6e5b8-2966-46d5-bcdb

In [35]:
# Step 6: Show results
print("\n 📊 Final reranked results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}: \n{doc.page_content}")


 📊 Final reranked results:

Rank 1: 
Nevertheless, rather than building all functionality into its core, Python was designed to be highly extensible through modules. This compact modularity has made it particularly popular as a means of

Rank 2: 
and logic programming.[61] Python is often referred to as a 'glue language'[62] because it is purposely designed to be able to integrate components written in other languages.

Rank 3: 
an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.

Rank 4: 
Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and

Rank 5: 
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indenta